# ComicInfo.xml Checker & Creator

Recursively scans `TARGET_DIR` for `.cbz` files and ensures each archive contains a `ComicInfo.xml` with:

- **Title** — filename without the extension and without the leading `[artist (circle)]` prefix
- **Series** — same as Title
- **Number** — always `1`
- **Writer** and **Penciller** — parent directory name (the artist directory), brackets removed
- **Genre** — `Doujinshi`

Archives missing `ComicInfo.xml` get one created; existing files have empty fields filled in (set `OVERWRITE_EXISTING = True` to always overwrite).

## Notes

- Python cannot read `smb://` URLs directly. Mount the share first with `gio mount smb://truenas.local/theia`, then set `TARGET_DIR` to the mount path (e.g. `/run/user/1000/gvfs/smb-share:server=truenas.local,share=theia/Misc`).
- Keep `DRY_RUN = True` to preview all changes before writing to any archive.
- Add artist directory names to `SKIP_DIRS` to exclude them entirely.
- Updating an existing `ComicInfo.xml` rewrites the archive (a `.tmp` file is written next to the CBZ, then moved over the original).

## Run order

1. Configuration
2. Helper functions
3. Scan & process


In [4]:
from pathlib import Path

# --- Configuration ---

# Python cannot read smb:// URLs directly. Mount the share first:
#     gio mount smb://truenas.local/theia
# then point TARGET_DIR at the mount location (adjust the UID if needed):
TARGET_DIR = Path("/run/user/1000/gvfs/smb-share:server=truenas.local,share=theia/Misc")

FILE_GLOB = "*.cbz"

# Preview changes without modifying any archives.
DRY_RUN = False

# Artist directories to skip entirely (matched case-insensitively against the
# directory directly above the CBZ file, with or without brackets).
SKIP_DIRS: set[str] = {"_Unknown_Artists_"}

# False = only fill fields that are missing/empty in existing ComicInfo.xml files.
# True  = always overwrite Title, Writer, Penciller, and Genre.
OVERWRITE_EXISTING = True

GENRE = "Doujinshi"


In [5]:
import re
import zipfile
from xml.etree import ElementTree

BRACKET_PREFIX = re.compile(r"^\s*\[[^\]]*\]\s*")


def clean_dir_name(dir_name: str) -> str:
    """Strip surrounding [brackets] from an artist directory name."""
    name = dir_name.strip()
    if name.startswith("[") and name.endswith("]"):
        name = name[1:-1].strip()
    return name


def parse_title(filename: str) -> str:
    """Derive the title from a filename minus the extension and [artist (circle)] prefix."""
    title = BRACKET_PREFIX.sub("", Path(filename).stem).strip()
    if not title:
        raise ValueError(f"could not derive a title from {filename!r}")
    return title


def metadata_for(cbz_path: Path) -> dict[str, str]:
    """Build the ComicInfo field values for one CBZ file."""
    artist = clean_dir_name(cbz_path.parent.name)
    title = parse_title(cbz_path.name)
    return {
        "Title": title,
        "Series": title,
        "Number": "1",
        "Writer": artist,
        "Penciller": artist,
        "Genre": GENRE,
    }


def build_comic_info(metadata: dict[str, str]) -> bytes:
    """Create a new UTF-8 ComicInfo.xml document."""
    root = ElementTree.Element("ComicInfo")
    for tag, text in metadata.items():
        ElementTree.SubElement(root, tag).text = text
    return ElementTree.tostring(root, encoding="utf-8", xml_declaration=True)


def update_comic_info(xml_bytes: bytes, metadata: dict[str, str]) -> tuple[bytes, list[str]]:
    """Fill empty fields (or overwrite them) in an existing ComicInfo.xml.

    Returns the updated XML plus a list of human-readable change descriptions.
    """
    root = ElementTree.fromstring(xml_bytes)
    changes: list[str] = []
    for tag, text in metadata.items():
        element = root.find(tag)
        current = (element.text or "").strip() if element is not None and element.text else ""
        if element is None:
            element = ElementTree.SubElement(root, tag)
        if (not current or OVERWRITE_EXISTING) and current != text:
            element.text = text
            changes.append(f"{tag}: {current!r} -> {text!r}" if current else f"{tag} = {text!r}")
    return ElementTree.tostring(root, encoding="utf-8", xml_declaration=True), changes

In [ ]:
def find_comic_info_entry(names: list[str]) -> str | None:
    """Return the archive entry whose basename is ComicInfo.xml, if present."""
    for name in names:
        if name.replace("\\", "/").rpartition("/")[2].lower() == "comicinfo.xml":
            return name
    return None


def rewrite_archive(cbz_path: Path, replace_entry: str | None, xml_bytes: bytes) -> None:
    """Rewrite the archive, replacing `replace_entry` (or adding ComicInfo.xml at the root)."""
    tmp_path = cbz_path.with_name(cbz_path.name + ".tmp")
    try:
        with zipfile.ZipFile(cbz_path, "r") as src, zipfile.ZipFile(
            tmp_path, "w", compression=zipfile.ZIP_DEFLATED
        ) as dst:
            for item in src.infolist():
                if replace_entry is not None and item.filename == replace_entry:
                    continue
                dst.writestr(item, src.read(item.filename))
            dst.writestr(replace_entry or "ComicInfo.xml", xml_bytes)
        tmp_path.replace(cbz_path)
    finally:
        tmp_path.unlink(missing_ok=True)


def process_cbz(cbz_path: Path) -> None:
    """Ensure one CBZ has a ComicInfo.xml with Title, Writer, Penciller, and Genre."""
    display = cbz_path.relative_to(TARGET_DIR)
    metadata = metadata_for(cbz_path)

    with zipfile.ZipFile(cbz_path, "r") as archive:
        entry = find_comic_info_entry(archive.namelist())
        existing = archive.read(entry) if entry is not None else None

    if existing is None:
        xml_bytes = build_comic_info(metadata)
        if DRY_RUN:
            print(f"[DRY RUN] would create ComicInfo.xml in {display}: {metadata}")
            return
        rewrite_archive(cbz_path, None, xml_bytes)
        print(f"[CREATED] {display}")
        return

    try:
        xml_bytes, changes = update_comic_info(existing, metadata)
    except ElementTree.ParseError as exc:
        print(f"[ERROR] {display}: unreadable ComicInfo.xml ({exc})")
        return

    if not changes:
        print(f"[OK] {display}")
    elif DRY_RUN:
        print(f"[DRY RUN] would update {display}: " + "; ".join(changes))
    else:
        rewrite_archive(cbz_path, entry, xml_bytes)
        print(f"[UPDATED] {display}: " + "; ".join(changes))


def is_skipped(cbz_path: Path, skip: set[str]) -> bool:
    """Check a CBZ's parent directory against the skip set, with or without brackets."""
    parent = cbz_path.parent.name
    return parent.casefold() in skip or clean_dir_name(parent).casefold() in skip


def main() -> None:
    """Scan TARGET_DIR and ensure every CBZ has a complete ComicInfo.xml."""
    if not TARGET_DIR.is_dir():
        raise NotADirectoryError(
            f"Target directory not found: {TARGET_DIR}\n"
            "Mount the share first, e.g.: gio mount smb://truenas.local/theia"
        )

    skip = {name.casefold() for name in SKIP_DIRS}
    skip |= {clean_dir_name(name).casefold() for name in SKIP_DIRS}

    all_cbz = sorted(p for p in TARGET_DIR.rglob(FILE_GLOB) if p.is_file())
    cbz_files = [p for p in all_cbz if not is_skipped(p, skip)]
    print(
        f"Found {len(cbz_files)} CBZ file(s) under {TARGET_DIR} "
        f"({len(all_cbz) - len(cbz_files)} skipped by directory)"
    )

    for cbz_path in cbz_files:
        try:
            process_cbz(cbz_path)
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            print(f"[ERROR] {cbz_path}: {exc}")


main()
